In [ ]:
!pip install pandas pysqlite3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 23.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import sqlite3
from google.colab import files

uploaded = files.upload()

Saving Employee_Sample_Data.csv to Employee_Sample_Data.csv


In [ ]:
df = pd.read_csv("Employee_Sample_Data.csv")
df.head(10)


,employee_id,full_name,job_title,department,business_unit,gender,ethnicity,age,hire_date,annual_salary,bonus,country,city
0,E02002,Kai Le,Controls Engineer,Engineering,Manufacturing,Male,Asian,47.0,2/5/2022,"$92,368",0%,United States,Columbus
1,E02003,Robert Patel,Analyst,Sales,Corporate,Male,Asian,58.0,10/23/2013,"$45,703",0%,United States,Chicago
2,E02004,Cameron Lo,Network Administrator,IT,Research & Development,Male,Asian,34.0,3/24/2019,"$83,576",0%,China,Shanghai
3,E02005,Harper Castillo,IT Systems Architect,IT,Corporate,Female,Latino,39.0,4/7/2018,"$98,062",0%,United States,Seattle
4,E02006,Harper Dominguez,Director,Engineering,Corporate,Female,Latino,42.0,6/18/2005,"$175,391",24%,United States,Austin
5,E02007,Ezra Vu,Network Administrator,IT,Manufacturing,Male,Asian,62.0,4/22/2004,"$66,227",0%,United States,Phoenix
6,E02008,Jade Hu,Sr. Analyst,Accounting,Specialty Products,Female,Asian,58.0,6/27/2009,"$89,744",0%,China,Chongqing
7,E02009,Miles Chang,Analyst II,Finance,Corporate,Male,Asian,62.0,2/19/1999,"$69,674",0%,China,Chengdu
8,E02010,Gianna Holmes,System Administrator,IT,Manufacturing,Female,Caucasian,38.0,9/9/2011,"$97,630",0%,United States,Seattle
9,E02011,Jameson Thomas,Manager,Finance,Specialty Products,Male,Caucasian,52.0,2/5/2015,"$105,879",10%,United States,Miami


In [ ]:
df.dtypes

,0
employee_id,object
full_name,object
job_title,object
department,object
business_unit,object
gender,object
ethnicity,object
age,float64
hire_date,object
annual_salary,object


In [ ]:
from numpy import nan
def clean_Employee(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df = df.dropna()

    str_columns = ["full_name",
                   "job_title",
                   "department",
                   "business_unit",
                   "gender",
                   "ethnicity",
                   "country",
                   "city"]

    for col in str_columns:
      df[col] = df[col].str.lower()

    df['age'] = pd.to_numeric(df['age'],errors = 'coerce')

    df['hire_date'] = pd.to_datetime(df['hire_date'], errors="coerce")

    df['annual_salary'] = df['annual_salary'].str[1:].str.strip().str.replace(",","")
    df['annual_salary'] = pd.to_numeric(df['annual_salary'], errors='coerce')

    df['bonus'] = df['bonus'].str.replace("%","")
    df['bonus'] = pd.to_numeric(df['bonus'], errors='coerce')/100

    return df



Employee_clean = clean_Employee(df)
Employee_clean



,employee_id,full_name,job_title,department,business_unit,gender,ethnicity,age,hire_date,annual_salary,bonus,country,city
0,E02002,kai le,controls engineer,engineering,manufacturing,male,asian,47.0,2022-02-05,92368,0.00,united states,columbus
1,E02003,robert patel,analyst,sales,corporate,male,asian,58.0,2013-10-23,45703,0.00,united states,chicago
2,E02004,cameron lo,network administrator,it,research & development,male,asian,34.0,2019-03-24,83576,0.00,china,shanghai
3,E02005,harper castillo,it systems architect,it,corporate,female,latino,39.0,2018-04-07,98062,0.00,united states,seattle
4,E02006,harper dominguez,director,engineering,corporate,female,latino,42.0,2005-06-18,175391,0.24,united states,austin
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1257,E02250,mila han,manager,sales,manufacturing,female,asian,54.0,2009-11-14,128791,0.06,united states,miami
1258,E02251,genesis herrera,manager,it,research & development,female,latino,34.0,2015-10-03,126898,0.10,brazil,manaus
1259,E02252,olivia vazquez,network engineer,it,specialty products,female,latino,53.0,2020-04-13,93053,0.00,brazil,sao paulo
1260,E02253,leilani ng,systems analyst,it,corporate,female,asian,48.0,2011-09-19,50513,0.00,united states,seattle


In [ ]:
import time
from google import genai
from google.colab import userdata

# Re-initialize client to ensure it's active in this session
API_KEY = userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=API_KEY)

# Reset global counter for API calls
i = 1

def get_ai_summary_with_rate_limit(row):
    global i
    # Print progress to indicate activity
    print(f"Generating summary for row {i}...")
    i += 1

    row_text = f"Name: {row['full_name']}, Job: {row['job_title']}, Department: {row['department']},"\
    f"Business Unit: {row['business_unit']}, Gender: {row['gender']}, Ethnicity: {row['ethnicity']}, Age: {row['age']}" \
    f", Annual Salary: {row['annual_salary']}, Bonus: {row['bonus']}, Country: {row['country']}"

    prompt = f"Summarize the following database record into one short, 15 max words sentence for an 'ai_summary' column: {row_text}"

    # Implement a retry mechanism with a delay to handle rate limits
    max_attempts = 3
    for attempt in range(max_attempts):
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt
            )
            # Sleep for 12 seconds to stay within the 5 requests per minute limit (60s / 5 = 12s)
            time.sleep(12)
            return response.text.strip()
        except Exception as e:
            print(f"Attempt {attempt + 1} failed for row {i-1}: {e}")
            if "RESOURCE_EXHAUSTED" in str(e) or "429" in str(e):
                if attempt < max_attempts - 1:
                    print(f"Waiting for {15 * (attempt + 1)} seconds before retrying...")
                    time.sleep(15 * (attempt + 1)) # Increase sleep time on successive retries
                else:
                    print("Max retries reached. Skipping this row.")
                    return "Summary Generation Failed (API Limit)"
            else:
                # For other errors, return an error message
                return f"Summary Generation Failed ({str(e)})"
    return "Summary Generation Failed (Unknown Error)" # Should not be reached


print("--- Addressing API Rate Limits and Daily Quota ---")
print("The Google Colab free API for Gemini-2.5-Flash has two main limitations:")
print("1. A rate limit of 5 requests per minute.")
print("2. A daily quota of 20 requests.")
print(f"Your DataFrame has {len(Employee_clean)} rows, which exceeds the daily quota.")
print("To demonstrate the functionality within the free tier limits, I will process the first 20 rows.")
print("For processing more data, you would need to upgrade your API plan.")

# Process a subset (e.g., first 20 rows) to stay within the daily quota for demonstration
# Make sure to work on a copy to avoid modifying the original DataFrame unexpectedly if not desired.
Employee_clean_limited = Employee_clean.head(20).copy()

# Apply the function to the limited DataFrame
Employee_clean_limited['ai_summary'] = Employee_clean_limited.apply(get_ai_summary_with_rate_limit, axis=1)

print("\n--- Generated AI Summaries (First 20 Rows) ---")
print(Employee_clean_limited[['full_name', 'ai_summary']])

--- Addressing API Rate Limits and Daily Quota ---
The Google Colab free API for Gemini-2.5-Flash has two main limitations:
1. A rate limit of 5 requests per minute.
2. A daily quota of 20 requests.
Your DataFrame has 966 rows, which exceeds the daily quota.
To demonstrate the functionality within the free tier limits, I will process the first 20 rows.
For processing more data, you would need to upgrade your API plan.
Generating summary for row 1...
Generating summary for row 2...
Generating summary for row 3...
Generating summary for row 4...
Generating summary for row 5...
Generating summary for row 6...
Generating summary for row 7...
Generating summary for row 8...
Generating summary for row 9...
Generating summary for row 10...
Generating summary for row 11...
Generating summary for row 12...
Generating summary for row 13...
Generating summary for row 14...
Generating summary for row 15...
Generating summary for row 16...
Generating summary for row 17...
Generating summary for row

In [ ]:
import sqlite3

# Create a SQLite database in memory (or specify a file path for persistent storage)
conn = sqlite3.connect(':memory:') # Using ':memory:' creates a temporary database

# Load the Employee_clean_limited DataFrame into a SQL table
# The table name will be 'employees'
Employee_clean_limited.to_sql('employees', conn, if_exists='replace', index=False)

# Print all tables in the database to verify
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("Tables in the SQLite database:")
for table in tables:
    print(table[0])

# Close the connection
conn.close()

Tables in the SQLite database:
employees


In [ ]:
import sqlite3
import pandas as pd

# Create a SQLite database in memory
conn = sqlite3.connect(':memory:')

# Load the Employee_clean_limited DataFrame into a SQL table named 'employees'
Employee_clean_limited.to_sql('employees', conn, if_exists='replace', index=False)

# SQL query to calculate the average length of 'ai_summary'
sql_query = """
SELECT AVG(LENGTH(ai_summary)) AS average_ai_summary_length
FROM employees;
"""

# Execute the query using pandas.read_sql_query()
result_df = pd.read_sql_query(sql_query, conn)

# Print the result
print("Average text length per ai_summary:")
print(result_df)

# Close the connection
conn.close()

Average text length per ai_summary:
   average_ai_summary_length
0                      69.35


SQL analysis.

### 1. High-Level Architecture

The pipeline is designed to process raw employee data from a CSV file, clean and transform it, enrich it with AI-generated summaries, and then store it in a temporary SQL database for analytical queries. The entire process is executed within a Google Colab environment, leveraging `pandas` for data manipulation, `google.genai` for AI integration, and `sqlite3` for database operations.

*   **Input**: `Employee_Sample_Data.csv`
*   **Processing Environment**: Google Colab
*   **Core Libraries**: `pandas`, `sqlite3`, `google.genai`
*   **Output**: Cleaned DataFrame (`Employee_clean`), AI-summarized DataFrame (`Employee_clean_limited`), In-memory SQLite database.

### 2. ETL Steps

The pipeline follows a standard Extract, Transform, Load (ETL) pattern:

*   **Extract**: The raw `Employee_Sample_Data.csv` file is uploaded and read into a pandas DataFrame (`df`).

*   **Transform**: A `clean_Employee` function is applied to the raw DataFrame to perform the following cleaning and transformation steps:
    *   **Missing Values**: Rows containing any missing values are dropped (`df.dropna()`).
    *   **Standardization**: Specified string columns (`full_name`, `job_title`, `department`, `business_unit`, `gender`, `ethnicity`, `country`, `city`) are converted to lowercase.
    *   **Type Conversion**:
        *   `age` is converted to a numeric type.
        *   `hire_date` is converted to datetime objects.
        *   `annual_salary` is cleaned by removing '$' and ',' characters, then converted to a numeric type.
        *   `bonus` is cleaned by removing '%' and then converted to a numeric type, subsequently divided by 100 to represent a decimal.

*   **Load**: A limited subset of the cleaned DataFrame (`Employee_clean_limited` - currently the first 20 rows due to API constraints) is loaded into an in-memory SQLite database as a table named `employees`.

### 3. Role of GenAI

Generative AI, specifically the `gemini-2.5-flash` model via the Google Colab free API, is utilized to generate a concise summary for each employee record. This summary is stored in a new `ai_summary` column.

*   **Function**: The `get_ai_summary_with_rate_limit` function constructs a prompt based on key employee attributes and sends it to the GenAI model.
*   **Constraint Handling**: Due to the Google Colab free API tier's limitations (5 requests per minute, 20 requests per day):
    *   A `time.sleep(12)` is implemented after each API call to respect the rate limit.
    *   Only the first 20 rows of the `Employee_clean` DataFrame are processed (`Employee_clean_limited`) to stay within the daily quota for demonstration purposes.
*   **Summarization**: Each row is summarized into a short sentence (max 15 words) providing a quick overview of the employee's profile.

### 4. SQL Analysis Summary

After loading the AI-summarized data into an SQLite database, a simple analytical query was performed:

*   **Query**: `SELECT AVG(LENGTH(ai_summary)) AS average_ai_summary_length FROM employees;`
*   **Purpose**: This query calculates the average character length of the AI-generated summaries across the `employees` table.
*   **Result**: The average length of the `ai_summary` column was found to be `69.35` characters.

### 5. Limitations & Future Improvements

#### Limitations:
*   **API Constraints**: The primary limitation is the free tier of the Google Colab GenAI API, which severely restricts the number of requests (20 daily, 5 per minute), making full dataset processing infeasible without an upgraded plan.
*   **In-Memory Database**: The SQLite database is in-memory, meaning all data and schema are lost upon Colab session termination. This is not suitable for persistent storage or production environments.
*   **Limited Data Processing**: Only 20 rows were processed by the GenAI component, which is a small fraction of the original dataset.

#### Future Improvements:
*   **Scalability for GenAI**: To process the entire dataset, a paid GenAI API plan would be required. Alternatively, for large datasets, a more sophisticated batch processing strategy with rate limit management and persistent queuing could be implemented.
*   **Persistent Data Storage**: Migrate from an in-memory SQLite database to a file-based SQLite database or a more robust relational database (e.g., PostgreSQL, MySQL) for data persistence and more complex querying capabilities.
*   **Enhanced AI Summaries**: Explore fine-tuning the GenAI model or using more advanced prompting techniques to generate even more insightful or specific summaries based on business needs.
*   **Automated Data Pipelines**: Integrate this pipeline into a scheduled workflow using tools like Apache Airflow or Google Cloud Composer for automated execution and monitoring.
*   **Dashboarding & Visualization**: Connect the processed data to business intelligence tools (e.g., Tableau, Power BI, Looker Studio) to create interactive dashboards for insights and reporting.